<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" width="100%" alt="OPIM 5512 banner"/>


# SHAP from scratch — on a tree you can *see*

**Dr. Dave Wanik · OPIM 5512: Data Science Using Python · University of Connecticut**

---

In the last notebook we computed SHAP by hand on a whiteboard model, then peeked at a **random forest** (100 trees — a black box). This time we go all the way: we fit **one very shallow decision tree** on California housing, and then show that **every SHAP number traces back to splits you can point at**. No magic.

By the end you will have, with nothing but `numpy` and `itertools`:

1. **Every coalition** — all $2^3=8$ "sub-models" and their values $v(S)$, in one table.
2. The **background** and the **base value** $v(\varnothing)=E[f(x)]$ — what "absent" really means.
3. A single row's SHAP values that satisfy **base $+\;\sum \phi =$ prediction**, exactly.
4. A **waterfall** for that one prediction (built by hand *and* with the `shap` library).
5. **Global from local** — average $|\phi|$ over many rows to recover feature importance.
6. A proof that the **`shap` library returns the same numbers** we did by hand.

> The whole point of using a *shallow* tree: it is simple enough that you can check the machine's arithmetic yourself.


## Setup

Runs top-to-bottom in Colab. The only install is `shap` (used right at the end to confirm our hand math).


In [ ]:
import itertools, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.metrics import r2_score

# --- reproducibility: fix every seed so the numbers are identical each run ---
RNG = 42        # train/test split + the decision tree
BG_SEED = 0     # which background rows we sample
np.random.seed(RNG)
print("ready — seeds fixed (RNG=42, BG_SEED=0). We install `shap` later, in section 9.")

## 1. A deliberately simple model

California Housing ships inside scikit-learn. We keep **three raw (unscaled) features** so the tree's split thresholds read in real units, and we cap the tree at **depth 3** so the whole model fits on a slide.

| feature | meaning |
|---|---|
| `MedInc` | median income in the block (in \$10,000s) |
| `AveRooms` | average rooms per household |
| `Latitude` | how far north (bigger = further north) |

The target `MedHouseVal` is the median home value in **\$100,000s**.


In [ ]:
data = fetch_california_housing(as_frame=True).frame
y = data["MedHouseVal"]
FEATURES = ["MedInc", "AveRooms", "Latitude"]
X = data[FEATURES]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RNG)

# shallow tree, raw features -> readable thresholds
tree = DecisionTreeRegressor(max_depth=3, random_state=RNG).fit(X_train, y_train)
print("Test R2:", round(r2_score(y_test, tree.predict(X_test)), 3))
print("\nWhich features did the tree actually split on?")
for f, imp in zip(FEATURES, tree.feature_importances_):
    print(f"  {f:<9} importance = {imp:.3f}")

## 2. Look at the whole model

Because it is shallow, we can print and draw the entire tree. Every prediction is just "follow the yes/no questions to a leaf." Read a couple of paths — this is the ground truth we will reconcile SHAP against.


In [ ]:
print(export_text(tree, feature_names=FEATURES))

plt.figure(figsize=(15, 6))
plot_tree(tree, feature_names=FEATURES, filled=True, rounded=True, fontsize=9, precision=2)
plt.title("The entire model — one shallow decision tree")
plt.show()

## 3. What does it mean for a feature to be "absent"? — the background

SHAP asks: *for this row, how much did each feature push the prediction away from normal?* To measure that we need to score **coalitions** — teams of features that are "known," with the rest "unknown."

The honest way to handle an **unknown** feature is **not** to plug in a zero. It is to **marginalize**: replace the unknown features with values drawn from a **background sample** of real rows, and **average** the model's prediction. Formally, for a coalition $S$ (the "known" features):

$$ v(S) \;=\; \frac{1}{N}\sum_{j=1}^{N} f\big(\underbrace{x_S}_{\text{this row on }S},\; \underbrace{X^{(j)}_{\bar S}}_{\text{background row }j\text{ on the rest}}\big) $$

Two anchors fall right out of this:

- $v(\varnothing)$ = average prediction with **nothing** known = the **base value** $E[f(x)]$.
- $v(\text{all})$ = prediction with **everything** known = the model's actual output for the row.

SHAP explains the gap between them.


In [ ]:
# background = a sample of TRAIN rows (this is the reference distribution).
# We use 100 rows on purpose: that's the shap library's default cap, so later
# our hand numbers and shap's will line up to the decimal (no silent subsampling).
background = X_train.sample(100, random_state=BG_SEED).reset_index(drop=True)

def v(S, x):
    '''Value of coalition S for row x: freeze S at x's values, borrow the rest
       from every background row, and average the tree's predictions.'''
    Z = background.copy()
    for f in S:
        Z[f] = x[f]
    return tree.predict(Z).mean()

# the ONE row we will explain all the way through
x = X_test.iloc[87]
base_value = v([], x)                       # v(empty) = E[f(x)]
prediction = tree.predict(x.to_frame().T)[0]

print("Row we're explaining (test #87):")
for f in FEATURES:
    print(f"   {f:<9} = {x[f]:.3f}")
print(f"\nbase value  v(empty) = E[f(x)] = {base_value:.4f}   ($100,000s)")
print(f"tree prediction f(x)          = {prediction:.4f}")
print(f"gap SHAP must explain         = {prediction - base_value:+.4f}")

## 4. Every coalition ("all the models")

There are $2^3 = 8$ possible teams of features. Each one is effectively a **different sub-model** — the tree, but only allowed to "know" the features in $S$. Here is the value of every one for our row.


In [ ]:
rows = []
for k in range(len(FEATURES) + 1):
    for S in itertools.combinations(FEATURES, k):
        rows.append({"coalition S": "{" + ", ".join(S) + "}" if S else "{ } (empty)",
                     "|S|": len(S),
                     "v(S)": round(v(list(S), x), 4)})
coalition_table = pd.DataFrame(rows)
print("The 8 coalition sub-models for test row #87:\n")
print(coalition_table.to_string(index=False))
print(f"\n(top row is the base value {base_value:.4f}; bottom row is the prediction {prediction:.4f})")

## 5. The Shapley value, from the coalitions

A feature's SHAP value is its **average marginal contribution** — how much it adds when it joins a team — averaged over **every** team it could join. The coalition form:

$$ \phi_i \;=\; \sum_{S \subseteq N\setminus\{i\}} \underbrace{\frac{|S|!\,(p-|S|-1)!}{p!}}_{\text{weight } w(S)} \Big[\, v(S\cup\{i\}) - v(S) \,\Big] $$

with $p=3$ features. The weight is just *the fraction of arrival orders in which exactly the players in $S$ show up before $i$*. For $p=3$: $w=\tfrac13$ when $|S|=0$ or $2$, and $w=\tfrac16$ when $|S|=1$.

We print the **full breakdown** so you can see each term — the same table you'd fill in by hand.


In [ ]:
def shapley_coalitions(x, verbose_for=None):
    p = len(FEATURES)
    phi = {}
    for i in FEATURES:
        others = [f for f in FEATURES if f != i]
        total = 0.0
        if verbose_for == i:
            print(f"feature {i}:")
        for k in range(len(others) + 1):
            for S in itertools.combinations(others, k):
                w = math.factorial(len(S)) * math.factorial(p - len(S) - 1) / math.factorial(p)
                marg = v(list(S) + [i], x) - v(list(S), x)
                total += w * marg
                if verbose_for == i:
                    lab = "{" + ", ".join(S) + "}" if S else "{ }"
                    print(f"   S={lab:<22} w={w:.3f}  v(S+i)-v(S)={marg:+.4f}  ->  {w*marg:+.4f}")
        if verbose_for == i:
            print(f"   phi[{i}] = {total:+.4f}\n")
        phi[i] = total
    return phi

# show the full worked breakdown for MedInc, then all three values
_ = shapley_coalitions(x, verbose_for="MedInc")
phi = shapley_coalitions(x)
print("SHAP values (from scratch):")
for f in FEATURES:
    print(f"   phi[{f:<9}] = {phi[f]:+.4f}")

### The same answer, the intuitive way (arrival orders)

The coalition weights look fussy. Here is the equivalent definition that most people find clearer: line the features up in **every possible order**, and each time a feature "walks in," record how much the prediction jumps. Average those jumps. With 3 features there are $3! = 6$ orders. It gives the **identical** numbers.


In [ ]:
def shapley_orderings(x):
    phi = {f: 0.0 for f in FEATURES}
    orders = list(itertools.permutations(FEATURES))
    for order in orders:
        team = []
        for f in order:
            before = v(team, x)
            team = team + [f]
            phi[f] += v(team, x) - before
    return {f: phi[f] / len(orders) for f in FEATURES}

phi_orders = shapley_orderings(x)
print("coalition form vs ordering form:")
for f in FEATURES:
    print(f"   {f:<9}  coalitions {phi[f]:+.4f}   orderings {phi_orders[f]:+.4f}   match={abs(phi[f]-phi_orders[f])<1e-9}")

## 6. The books balance — local accuracy

The defining promise of SHAP: **base value + the SHAP values = the prediction**, exactly. Nothing invented, nothing lost.


In [ ]:
total = base_value + sum(phi.values())
print(f"base value           = {base_value:.4f}")
for f in FEATURES:
    print(f"  + SHAP[{f:<9}]   = {phi[f]:+.4f}")
print("-" * 34)
print(f"= {total:.4f}   vs prediction {prediction:.4f}   (match = {abs(total-prediction)<1e-9})")

## 7. The waterfall — one prediction, explained

Start at the base value and let each feature push the bar up (green) or down (red) until we land on the prediction. This is *the* SHAP picture for a single row — and we can draw it from our own numbers.


In [ ]:
order = sorted(FEATURES, key=lambda f: -abs(phi[f]))   # biggest push first
fig, ax = plt.subplots(figsize=(9, 4.5))
running = base_value
ax.axhline(base_value, color="grey", ls="--", lw=1)
ax.text(-0.45, base_value, f"base\n{base_value:.2f}", va="center", ha="right", fontsize=10)
for k, f in enumerate(order):
    c = phi[f]
    ax.bar(k, c, bottom=running, width=0.6,
           color="#2ca02c" if c >= 0 else "#d62728", edgecolor="black")
    ax.text(k, running + c/2, f"{f}\n{c:+.2f}", ha="center", va="center",
            color="white", fontweight="bold", fontsize=9)
    running += c
ax.axhline(running, color="black", lw=1)
ax.text(len(order)-0.55, running, f"pred\n{running:.2f}", va="center", ha="left", fontsize=10)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order)
ax.set_ylabel("median value ($100,000s)")
ax.set_title(f"Why the tree predicted {prediction:.2f} for test house #87")
plt.tight_layout(); plt.show()

### Read the story (and spot the interaction)

For this house you should see something like:

- **`MedInc` pushes down hard.** Income is low ($\approx$ \$16,600), and income is the tree's first and most-used split, so it drags the prediction well below average.
- **`AveRooms` pushes up.** More rooms than the neighborhood norm lifts the value a bit.
- **`Latitude` nudges down** — a modest location effect.

Look back at the breakdown table for `Latitude`: its marginal contribution is **exactly 0** in every coalition that already contains `MedInc`. That is a real **interaction** the tree learned — once income has sent the house down a particular branch, latitude no longer changes the leaf. SHAP still gives latitude fair partial credit from the coalitions where income is *absent*. This is the kind of nuance a single "importance" number throws away.


## 8. Global importance from local explanations

Compute SHAP for **many** rows and average the **absolute** values per feature. That mean-$|\phi|$ ranking is a *global* importance measure — assembled entirely from per-row explanations (exactly what `shap`'s bar/beeswarm summary shows).


In [ ]:
N = 150
sample = X_test.iloc[:N]
shap_matrix = np.array([[shapley_coalitions(row)[f] for f in FEATURES]
                        for _, row in sample.iterrows()])
mean_abs = np.abs(shap_matrix).mean(axis=0)

o = np.argsort(mean_abs)
plt.figure(figsize=(7, 3.5))
plt.barh(np.array(FEATURES)[o], mean_abs[o], color="#1f77b4")
plt.xlabel("mean |SHAP|  (average push size, $100,000s)")
plt.title(f"Global feature importance from {N} local explanations")
plt.tight_layout(); plt.show()
for f, m in sorted(zip(FEATURES, mean_abs), key=lambda t: -t[1]):
    print(f"   {f:<9} mean|SHAP| = {m:.4f}")

## 9. Does the real `shap` library agree?

Everything so far used only `numpy` and `itertools`. The famous `shap` package does **exactly** this calculation, just with clever shortcuts. To match our numbers we point it at the **same background** and ask for **interventional** values.

> **Why this flag matters (background SHAP!).** `shap.TreeExplainer` has two modes. The default, `tree_path_dependent`, estimates "absent" features from how training rows *flowed down the tree* — it needs **no background** and can give different numbers. The `interventional` mode with an explicit `data=background` uses the **same marginalize-over-a-background** value function we built by hand — so it lines up to the decimal. Choosing the background **is** a modeling choice, and it sets your base value.


In [ ]:
# install shap right where we first need it (Colab)
try:
    import shap
except Exception:
    !pip -q install shap
    import shap
print("shap", shap.__version__)

In [ ]:
explainer = shap.TreeExplainer(tree, data=background, feature_perturbation="interventional")
expl = explainer(X_test.iloc[:N])   # Explanation object

print("Row #87 — ours vs shap:")
for f in FEATURES:
    ours = phi[f]
    theirs = expl.values[87][FEATURES.index(f)]
    print(f"   {f:<9} ours {ours:+.4f}   shap {theirs:+.4f}   match={abs(ours-theirs)<1e-6}")
print(f"\nshap base value {expl.base_values[0]:.4f}   ours {base_value:.4f}")

In [ ]:
# shap's own waterfall for the same row — compare to ours in section 7
shap.plots.waterfall(expl[87], show=True)

In [ ]:
# and the beeswarm: every dot is one house's SHAP value for that feature
shap.plots.beeswarm(expl, show=True)

## 10. Background matters — a quick experiment

The base value is *defined by* the background. Swap the 200-row sample for a **single average row** and watch the base value (and the SHAP split) shift — same machinery, different reference point. There is no single "right" background; it is a choice you make and should be able to explain.


In [ ]:
mean_row = X_train.mean()
bg_options = {
    "100 random train rows": X_train.sample(100, random_state=BG_SEED).reset_index(drop=True),
    "1 average row":         mean_row.to_frame().T.reset_index(drop=True),
}
for name, bg in bg_options.items():
    def v_bg(S, x, bg=bg):
        Z = bg.copy()
        for f in S:
            Z[f] = x[f]
        return tree.predict(Z).mean()
    p = len(FEATURES); phi_bg = {}
    for i in FEATURES:
        others = [f for f in FEATURES if f != i]; tot = 0.0
        for k in range(len(others)+1):
            for S in itertools.combinations(others, k):
                w = math.factorial(len(S))*math.factorial(p-len(S)-1)/math.factorial(p)
                tot += w*(v_bg(list(S)+[i], x) - v_bg(list(S), x))
        phi_bg[i] = tot
    base_bg = v_bg([], x)
    print(f"background = {name:<22}  base={base_bg:.3f}  " +
          "  ".join(f"{f}={phi_bg[f]:+.3f}" for f in FEATURES) +
          f"  (base+sum={base_bg+sum(phi_bg.values()):.3f})")

## Wrap-up

> **Key takeaways**
> - A coalition value $v(S)$ is the tree's prediction when only $S$ is "known" and the rest is **averaged over a background**. All $2^p$ of them are just sub-models.
> - The **base value** is $v(\varnothing)=E[f(x)]$ — set entirely by your **background** choice.
> - A feature's SHAP value is its **average marginal contribution** over all coalitions/orders (two formulas, one answer).
> - **Local accuracy**: base $+\sum\phi=$ prediction, exactly — we watched the books balance.
> - **Global from local**: mean $|\phi|$ over many rows is a feature-importance ranking.
> - The `shap` library reproduces our hand numbers when it uses the **same interventional background** — and hands us the polished **waterfall** and **beeswarm** for free.

**In Lab 2** you'll run `shap` on a stronger model and read these same plots to defend a prediction. You now know exactly what every bar means, because you built it from scratch.

**Practice by hand:** work the companion worksheet **`SHAP_ByHand`** (a 3-feature coalition table with clean numbers) before lab.
